In [ ]:

import chess

def get_legal_moves_for_state(move_sequence):
    """
    Replays a game from a list of SAN moves and returns the 
    legal moves available at that specific board state.
    """
    board = chess.Board()
    
    # Replay the move sequence
    for move in move_sequence:
        board.push_san(move)
        
    # Get all legal moves in SAN format (e.g., 'Nf3', 'axb5', 'O-O')
    # We use board.san(move) to convert the move object to text
    legal_moves_san = [board.san(move) for move in board.legal_moves]
    
    return board, legal_moves_san

# --- Test Case 1: Ruy Lopez (Open Game) - 11 Moves Deep (5.5 turns) ---
moves_ruy_lopez = [
    "e4", "e5", 
    "Nf3", "Nc6", 
    "Bb5", "a6", 
    "Ba4", "Nf6", 
    "O-O", "Be7", 
    "Re1"
]

# --- Test Case 2: Sicilian Defense (Semi-Open) - 12 Moves Deep (6 turns) ---
moves_sicilian = [
    "e4", "c5", 
    "Nf3", "d6", 
    "d4", "cxd4", 
    "Nxd4", "Nf6", 
    "Nc3", "a6", 
    "Bg5", "e6"
]

# --- Test Case 3: Queen's Gambit Declined (Closed) - 10 Moves Deep (5 turns) ---
moves_qgd = [
    "d4", "d5", 
    "c4", "e6", 
    "Nc3", "Nf6", 
    "Bg5", "Be7", 
    "e3", "O-O"
]

# List of cases to iterate through
test_cases = [
    ("Ruy Lopez", moves_ruy_lopez),
    ("Sicilian Defense", moves_sicilian),
    ("Queen's Gambit", moves_qgd)
]

# Execute and Print Results
for name, moves in test_cases:
    board, legal_moves = get_legal_moves_for_state(moves)
    
    print(f"")
    print(f"Last Move Played: {moves[-1]}")
    print("Current Board State:")
    print(board)
    print(f"Legal Moves: {legal_moves}...")
    print("-" * 40 + "\n")



Current Board State:
r . b q k . . r
. p p p b p p p
p . n . . n . .
. . . . p . . .
B . . . P . . .
. . . . . N . .
P P P P . P P P
R N B Q R . K .
Legal Moves: ['Rg8', 'Rf8', 'Kf8', 'Rb8', 'Ra7', 'Bf8', 'Bd6', 'Bc5', 'Bb4', 'Ba3', 'Ng8', 'Nh5', 'Nd5', 'Ng4', 'Nxe4', 'Nb8', 'Na7', 'Na5', 'Nd4', 'Nb4', 'O-O', 'h6', 'g6', 'd6', 'b6', 'a5', 'h5', 'g5', 'd5', 'b5']...
----------------------------------------



Current Board State:
r n b q k b . r
. p . . . p p p
p . . p p n . .
. . . . . . B .
. . . N P . . .
. . N . . . . .
P P P . . P P P
R . . Q K B . R
Legal Moves: ['Bh6', 'Bxf6', 'Bh4', 'Bf4', 'Be3', 'Bd2', 'Bc1', 'Nxe6', 'Nc6', 'Nf5', 'Ndb5', 'Nf3', 'Nb3', 'Nde2', 'Nd5', 'Ncb5', 'Na4', 'Nce2', 'Nb1', 'Rg1', 'Bxa6', 'Bb5+', 'Bc4', 'Bd3', 'Be2', 'Ke2', 'Kd2', 'Qh5', 'Qg4', 'Qf3', 'Qd3', 'Qe2', 'Qd2', 'Qc1', 'Qb1', 'Rc1', 'Rb1', 'e5', 'h3', 'g3', 'f3', 'b3', 'a3', 'h4', 'g4', 'f4', 'b4', 'a4']...
----------------------------------------



Current Board State:
r n b q . r k .
p p p 

In [16]:
import boto3

dynamodb = boto3.resource(
    'dynamodb',
    endpoint_url='http://localhost:8000',
    region_name='us-east-1',
    aws_access_key_id='anything',
    aws_secret_access_key='anything'
)

def setup_database():
    try:
        table = dynamodb.create_table(
            TableName='ChessMoves',
            KeySchema=[
                {'AttributeName': 'PK', 'KeyType': 'HASH'},  # Partition Key
                {'AttributeName': 'SK', 'KeyType': 'RANGE'} # Sort Key
            ],
            AttributeDefinitions=[
                {'AttributeName': 'PK', 'AttributeType': 'S'},
                {'AttributeName': 'SK', 'AttributeType': 'S'}
            ],
            ProvisionedThroughput={
                'ReadCapacityUnits': 5,
                'WriteCapacityUnits': 5
            }
        )
        table.meta.client.get_waiter('table_exists').wait(TableName='ChessMoves')
    except Exception as e:
        if "ResourceInUseException" in str(e):
            print("Table already exists. You are good to go!")
        else:
            print(f"Error: {e}")

setup_database()

table = dynamodb.Table('ChessMoves')

Table already exists. You are good to go!


In [17]:

import json
import boto3
from decimal import Decimal
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

dynamodb = boto3.resource(
    'dynamodb',
    endpoint_url='http://localhost:8000',
    region_name='us-east-1',
    aws_access_key_id='anything',
    aws_secret_access_key='anything'
)
table = dynamodb.Table('ChessMoves')

def process_and_store_moves(json_input, game_id, fen):
    if isinstance(json_input, str):
        moves_data = json.loads(json_input)
    else:
        moves_data = json_input

    descriptions = [item['desc'] for item in moves_data]
    
    vectors = model.encode(descriptions)

    with table.batch_writer() as batch:
        for i, item in enumerate(moves_data):
            move_san = item['move']
            description = item['desc']
            
            vector_as_decimal = [Decimal(str(v)) for v in vectors[i].tolist()]
            
            db_item = {
                'PK': f"GAME#{game_id}#FEN#{fen}",
                'SK': f"MOVE#{move_san}",
                'move_san': move_san,
                'description': description,
                'embedding': vector_as_decimal
            }
            
            batch.put_item(Item=db_item)
    
    print(f"Stored {len(moves_data)} moves using Hugging Face embeddings.")


def find_move_from_natural_language(user_query, game_id, fen):
    query_vector = model.encode(user_query)

    response = table.query(
        KeyConditionExpression=boto3.dynamodb.conditions.Key('PK').eq(f"GAME#{game_id}#FEN#{fen}")
    )
    items = response.get('Items', [])

    if not items:
        return None

    best_move = None
    highest_score = -1

    for item in items:
        stored_vector = np.array([float(v) for v in item['embedding']])
        
        score = np.dot(query_vector, stored_vector) / (np.linalg.norm(query_vector) * np.linalg.norm(stored_vector))
        
        if score > highest_score:
            highest_score = score
            best_move = item['move_san']

    return best_move

with open('sample1.json', 'r') as f:
    llm_json = json.load(f)

game_id = "123"
current_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"

# Store them
process_and_store_moves(llm_json, game_id, current_fen)

# Simulate User Query
user_input = "Push my pawn on the left side to get more space"
matched_move = find_move_from_natural_language(user_input, game_id, current_fen)

print(f"User said: '{user_input}'")
print(f"Matched Move: {matched_move}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 647.29it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stored 43 moves using Hugging Face embeddings.
User said: 'Push my pawn on the left side to get more space'
Matched Move: c5


In [ ]:
import chess
import json
import numpy as np
from decimal import Decimal
from sentence_transformers import SentenceTransformer

def generate_templates(board, move, san) -> list[str]:
    piece = board.piece_at(move.from_square)
    target = chess.square_name(move.to_square)
    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None
    templates = []
    file_name = chess.FILE_NAMES[chess.square_file(move.from_square)]

    if is_castle:
        if chess.square_file(move.to_square) == 6:
            templates += ["Castle kingside", "Kingside castle", "Perform kingside castling",
                          "King castles short", "Castle short", "Short castle"]
        else:
            templates += ["Castle queenside", "Queenside castle", "Perform queenside castling",
                          "King castles long", "Castle long", "Long castle"]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [f"Promote pawn to {promo_piece} on {target}",
                      f"Advance pawn to {target} and promote to {promo_piece}",
                      f"Move pawn to {target} and make it a {promo_piece}"]
        return templates

    p_name = chess.piece_name(piece.piece_type)
    if is_capture:
        templates += [f"{p_name.capitalize()} captures on {target}",
                      f"Capture the piece on {target} with the {p_name}",
                      f"Take on {target} using the {p_name}",
                      f"Use the {p_name} to capture on {target}"]
    elif piece.piece_type == chess.PAWN:
        templates += [f"Advance pawn to {target}", f"Push the pawn to {target}",
                      f"Move pawn to {target}", f"Advance {file_name} pawn"]
    else:
        templates += [f"Move the {p_name} to {target}", f"Play {p_name} to {target}",
                      f"Develop the {p_name} to {target}", f"{p_name.capitalize()} goes to {target}"]
    templates.append(f"Play {san}")
    return templates

def get_move_descriptions(board: chess.Board) -> list[dict]:
    """
    Takes a board, returns all (san, desc) pairs for every legal move —
    the format your embedding pipeline expects.
    """
    records = []
    for move in board.legal_moves:
        san = board.san(move)
        for desc in generate_templates(board, move, san):
            records.append({"move": san, "desc": desc})
    return records


model = SentenceTransformer('all-MiniLM-L6-v2')

def build_index(records: list[dict]) -> tuple[list, np.ndarray]:
    """
    Embeds all descriptions and returns:
      - the original records list  (so you can look up move_san by index)
      - a matrix of embeddings    (shape: N x embedding_dim)
    """
    descriptions = [r["desc"] for r in records]
    embeddings = model.encode(descriptions, convert_to_numpy=True)
    return records, embeddings


def find_best_move(user_query: str, records: list[dict], embeddings: np.ndarray) -> str:
    # cosine similarity search: embed the query, then dot product against all rows

    query_vec = model.encode(user_query, convert_to_numpy=True)
    # cosine similarity against every row
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_vec)
    scores = embeddings @ query_vec / norms
    best_idx = int(np.argmax(scores))
    return records[best_idx]["move"]


def process_board_state(board: chess.Board, user_query: str) -> str:
    """
    All-in-one function Thomas can call in his game loop:
      board       – the current chess.Board object
      user_query  – whatever the user typed

    Returns the matched SAN move string (e.g. "Nf3", "O-O", "exd5").
    """
    records = get_move_descriptions(board)     
    records, embeddings = build_index(records) 
    return find_best_move(user_query, records, embeddings) 